# 57. Full-range 6D calibration regression on KAIST

Adapted from Notebook **47** (corrected left/right LiDAR clock references, same 1-IMU/2-LiDAR topology, data splits, physically consistent true-calibration augmentation and checkpoint infrastructure).

**Experiment:** replace the legacy SE(3) log-vector head with direct **6D SO(3) rotation + 3D parent-frame translation + 1D time-offset regression**. The existing Transformer, calibration pooler and shared MLP are initialized from a compatible checkpoint; the 10-output head is new. **All parameters train immediately**, with the Transformer learning rate 10× smaller than the head/MLP learning rate. No FM integration, no 30° bounded teacher and **no topology augmentation**.

The notebook adds a *reversible stochastic preflight*, fixed held-out angular probes (0°, 30°, 90°, 150°, 175°, 179°), per-angle plots at every training epoch, and a final paired baseline comparison. The **test split remains untouched** until final evaluation is explicitly enabled.

**Important experimental change vs Notebook 47:** Prior calibration faults now change rotation, translation and time offset **independently**. A pure rotational perturbation no longer rotates the translation vector through a left SE(3) exponential. This matches the direct 6D model's update convention and avoids large artificial translation targets at coarse angles. True-calibration augmentation itself is reused unchanged.

## 1. Imports, repository root and 6D module availability

In [ ]:
from pathlib import Path
from dataclasses import asdict, replace
from collections import defaultdict
import gc
import math
import random
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from IPython.display import display

ROOT = Path.cwd()
for candidate in (ROOT, *ROOT.parents):
    if (candidate / 'src' / 'obscalib').is_dir():
        ROOT = candidate
        break
else:
    raise FileNotFoundError('Run from the nn-obs-calib repository or a descendant directory.')
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from kaist_dataset.dataset import KAISTCalibrationDataset, KAISTRigCalibration, build_kaist_sequence_specs, make_kaist_dataloader
from obscalib.augmentations import PerturbationDistribution
from obscalib.augmentations.config import AugmentationConfig, CalibrationEventConfig, CommonBodyFrameRandomizationConfig, FrameRandomizationConfig, NoiseAugmentationConfig, PerturbationMagnitudeConfig, PriorPerturbationConfig, SamplingRateAugmentationConfig, SensorTopologyAugmentationConfig, TrueCalibrationAugmentationConfig
from obscalib.augmentations.pipeline import AugmentationPipeline
from obscalib.augmentations.prior_perturbation import CalibrationPriorPerturber
from obscalib.config import CalibrationPoolerConfig, MLPConfig, PooledCalibrationHeadConfig, PooledCalibrationModelConfig, TransformerConfig, WindowingConfig
from obscalib.data.collation import collate_windows
from obscalib.data.device import window_batch_to
from obscalib.data.structures import WindowBatch
from obscalib.geometry.lie import so3_exp
from obscalib.geometry.processing import GeometryProcessor
from obscalib.geometry.rotation_6d import matrix_to_rotation_6d, rotation_6d_to_matrix
from obscalib.models.pooled_calibration_6d_model import PooledCalibration6DModel, load_6d_pretrained_features
from obscalib.calibration.rotation_6d_update import apply_calibration_6d_update
from obscalib.training.rotation_6d_loss import Calibration6DLossWeights, calibration_6d_loss
from obscalib.tokenization.tokenizer import Tokenizer
from obscalib.training.checkpoint import load_training_checkpoint, save_training_checkpoint

print('Repository:', ROOT, '| Torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())


## 2. Configuration and training/validation policy

In [ ]:
RUN_NAME = 'kaist_pooled_calibration_6d.57_full_rotation_regression'
RUN_DIR = ROOT / 'runs' / RUN_NAME
SOURCE_RUN_NAME = 'kaist_pooled_calibration_paired_002.24_mixed_imu_lidar_topology'
SOURCE_FEATURE_CHECKPOINT = ROOT / 'runs' / SOURCE_RUN_NAME / 'best.pt'
PRETRAIN_FEATURES = True  # Set False for an architecture-matched scratch control.

RANDOM_SEED, TRAIN_AUG_SEED, VAL_AUG_SEED, TEST_AUG_SEED = 12345, 23456, 34567, 45678
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DTYPE = torch.float32
CALIBRATION_KEYS = ('imu', 'lidar', 'lidar_right')
WINDOW_DURATION_S, WINDOW_STRIDE_S, MAX_SAMPLES_PER_SENSOR = 5.0, 5.0, 700
BATCH_SIZE, NUM_WORKERS, GRADIENT_ACCUMULATION_STEPS = 8, 0, 2
NUM_EPOCHS, WEIGHT_DECAY, GRADIENT_CLIP_NORM = 80, 1e-4, 1.0
HEAD_MLP_LR = 3e-5
TRANSFORMER_LR = HEAD_MLP_LR / 10.0
LR_REDUCE_FACTOR, LR_PATIENCE_EPOCHS = 0.5, 8
RUN_FULL_TRAINING = True
RESUME_FULL_TRAINING = False
MAX_TRAIN_BATCHES_PER_EPOCH, MAX_VAL_BATCHES_PER_EPOCH = None, None

MEASUREMENT_DIM, TOKEN_INPUT_DIM = 6, 8
D_MODEL, N_HEADS, NUM_TRANSFORMER_LAYERS, DIM_FEEDFORWARD = 128, 8, 12, 512
TRANSFORMER_DROPOUT, NUM_SUMMARY_TOKENS = 0.0, 1
POOLER_N_HEADS, POOLER_DROPOUT = 8, 0.0
MLP_HIDDEN_DIMS, SHARED_FEATURE_DIM, MLP_DROPOUT = (256, 128), 128, 0.0
HEAD_HIDDEN_DIMS, HEAD_DROPOUT = (128, 64), 0.0

# Original notebook 47 tau references (all three physical sensor streams remain present).
REFINED_TAU_BY_SENSOR_S = {'imu': 0.0, 'lidar': -0.025, 'lidar_right': -0.035}
TRUE_CALIBRATION_ROTATION_MAX_RAD = np.deg2rad(180.0)
TRUE_CALIBRATION_TIME_OFFSET_MAX_S = 0.030

# Fine prior distribution retains Notebook 47's Student-t rotation/timing model.
FINE_ROTATION_MAX_DEG, FINE_ROTATION_SCALE_DEG = 60.0, 30.0
# Coarse faults are deliberately broad, not Student-t tails that almost never visit 175–179°.
COARSE_ROTATION_MIN_DEG, COARSE_ROTATION_MAX_DEG = 60.0, 179.0
COARSE_PRIOR_PROBABILITY = 0.50
PRIOR_TRANSLATION_MAX_M, PRIOR_TRANSLATION_SCALE_M = 0.05, 0.02
PRIOR_TAU_MAX_S, PRIOR_TAU_SCALE_S = 0.50, 0.25
STUDENT_T_DOF = 3
ACTIVE_ROTATION_SUPERVISION_WEIGHT = 0.75
LOSS_WEIGHTS = Calibration6DLossWeights(rotation_weight=2.0, rotation_6d_weight=1.0, rotation_chordal_weight=1.0, translation_weight=1.0, time_offset_weight=1.0, translation_scale_m=0.05, time_offset_scale_s=0.05)

# Fixed validation probe: model-selected checkpoint must work at coarse angles, not only lower val loss.
PROBE_ANGLES_DEG = (0.0, 30.0, 90.0, 150.0, 175.0, 179.0)
PROBE_SCORE_ANGLES_DEG = (30.0, 90.0, 150.0, 175.0, 179.0)
PROBE_WINDOWS, PROBE_BATCH_SIZE, PROBE_AXIS_SEEDS = 16, 8, 1
PROBE_FAULT_KEYS = ('lidar', 'lidar_right')
PROBE_EVERY_EPOCHS = 1
ZERO_DRIFT_WEIGHT = 0.15  # penalty in relative-score units: zero-angle drift / 30°
SHOW_PLOTS = True

# Reversible preflight with independent stochastic physical windows/priors each update.
RUN_STOCHASTIC_PRECHECK = True
STOCHASTIC_STEPS, STOCHASTIC_BATCH_SIZE, STOCHASTIC_REPORT_EVERY = 250, 8, 25
STOCHASTIC_PROBE_WINDOWS = 8
SAVE_STOCHASTIC_CANDIDATE = True
LOAD_STOCHASTIC_CANDIDATE_FOR_FULL = False  # Explicit opt-in; by default restore pretrained initialization.

# Test data are kept untouched until consciously enabling final evaluation.
RUN_FINAL_TEST_EVALUATION = False

assert abs(TRANSFORMER_LR * 10.0 - HEAD_MLP_LR) < 1e-12
assert 0.0 < COARSE_ROTATION_MIN_DEG < COARSE_ROTATION_MAX_DEG < 180.0
assert 0 <= COARSE_PRIOR_PROBABILITY <= 1
assert set(PROBE_FAULT_KEYS).issubset(CALIBRATION_KEYS)
if PRETRAIN_FEATURES and not SOURCE_FEATURE_CHECKPOINT.is_file():
    raise FileNotFoundError(f'Pretrained checkpoint not found: {SOURCE_FEATURE_CHECKPOINT}')
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if DEVICE.type == 'cuda':
    torch.cuda.manual_seed_all(RANDOM_SEED)
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Run:', RUN_DIR)
print('Learning rates: head/MLP/pooler=', HEAD_MLP_LR, '| Transformer=', TRANSFORMER_LR)
print('Single topology:', CALIBRATION_KEYS, '| no topology or artificial sensor-noise augmentation')


## 3. KAIST data, splits and corrected timing reference (from Notebook 47)

In [ ]:
KAIST_DATASET_ROOT = Path("/mnt/d/Downloads/MobRobLab/KAISTDataset")
LIDAR_POSE_ROOT = Path("/home/camel/Skoltech/phd_proposal/data/KAISTDataset")

TRAIN_NAMES = ["Urban08", "Urban12", "Urban13", "Urban14", "Urban15", "Urban16"]
VAL_NAMES = ["Urban26", "Urban28"]
TEST_NAMES = ["Urban38", "Urban39"]
ALL_SEQUENCE_NAMES = TRAIN_NAMES + VAL_NAMES + TEST_NAMES

if len(ALL_SEQUENCE_NAMES) != len(set(ALL_SEQUENCE_NAMES)):
    raise ValueError("Train, validation, and test sequence sets overlap.")

T_B_L = np.array([
    [-0.514066, -0.702201, -0.492595, -0.440699],
    [0.486485, -0.711672, 0.506809, 0.397052],
    [-0.706447, 0.0208933, 0.707457, 1.90953],
    [0.0, 0.0, 0.0, 1.0],
], dtype=float)

# Right LiDAR -> body, in the same convention as T_B_L for the left LiDAR.
T_B_L_right = np.array([
    [-0.512152,  0.699241, -0.498761, -0.449885],
    [-0.494811, -0.714859, -0.494104, -0.416713],
    [-0.702041, -0.0062642, 0.712109,  1.91294],
    [ 0.0,       0.0,       0.0,       1.0],
], dtype=float)

T_I_B = np.array([
    [1.0, 0.0, 0.0, -0.07],
    [0.0, 1.0, 0.0, 0.0],
    [0.0, 0.0, 1.0, 1.7],
    [0.0, 0.0, 0.0, 1.0],
], dtype=float)

rig_calibration = KAISTRigCalibration(T_B_I=np.linalg.inv(T_I_B), T_B_L=T_B_L, tau_imu_s=KAIST_TAU_IMU_S, tau_lidar_s=KAIST_TAU_LIDAR_S, T_B_L_right=T_B_L_right, tau_lidar_right_s=KAIST_TAU_LIDAR_RIGHT_S)
windowing_config = WindowingConfig(window_duration_s=WINDOW_DURATION_S, window_stride_s=WINDOW_STRIDE_S, max_samples_per_sensor=MAX_SAMPLES_PER_SENSOR)

print("train:", TRAIN_NAMES)
print("validation:", VAL_NAMES)
print("test:", TEST_NAMES)
print("Corrected rig τ [ms]:", {key: 1000.0 * value for key, value in REFINED_TAU_BY_SENSOR_S.items()})


In [ ]:
all_specs = build_kaist_sequence_specs(sequence_names=ALL_SEQUENCE_NAMES, kaist_dataset_root=KAIST_DATASET_ROOT, lidar_pose_root=LIDAR_POSE_ROOT, lidar_right_pose_filename="lidar_map_poses_vlp_right.csv")

# The 3-sensor experiment must never silently fall back to 2 sensors.
for spec in all_specs:
    if spec.lidar_right_pose_csv is None or not spec.lidar_right_pose_csv.is_file():
        raise FileNotFoundError(f"Missing required right-LiDAR pose CSV for {spec.name}: {spec.lidar_right_pose_csv}")
spec_by_name = {spec.name: spec for spec in all_specs}

train_dataset = KAISTCalibrationDataset([spec_by_name[name] for name in TRAIN_NAMES], rig_calibration, windowing_config, measurement_dtype=torch.float32, source_timestamp_dtype=torch.float64, window_timestamp_dtype=torch.float32)
val_dataset = KAISTCalibrationDataset([spec_by_name[name] for name in VAL_NAMES], rig_calibration, windowing_config, measurement_dtype=torch.float32, source_timestamp_dtype=torch.float64, window_timestamp_dtype=torch.float32)
test_dataset = KAISTCalibrationDataset([spec_by_name[name] for name in TEST_NAMES], rig_calibration, windowing_config, measurement_dtype=torch.float32, source_timestamp_dtype=torch.float64, window_timestamp_dtype=torch.float32)

for split_name, dataset in (("train", train_dataset), ("validation", val_dataset), ("test", test_dataset)):
    if tuple(dataset.calibration_keys) != CALIBRATION_KEYS:
        raise AssertionError(f"{split_name} dataset has calibration keys {dataset.calibration_keys}, expected {CALIBRATION_KEYS}")

print(train_dataset.summary())
print()
print(val_dataset.summary())
print()
print(test_dataset.summary())

# Check the actual WindowBatch ground truth, not just constructor arguments.
for split_name, split_dataset in (("train", train_dataset), ("validation", val_dataset), ("test", test_dataset)):
    sample = collate_windows([split_dataset[0]])
    for sensor, expected_s in REFINED_TAU_BY_SENSOR_S.items():
        observed = float(sample.current_calibration[sensor].time_offset.reshape(-1)[0].item())
        assert abs(observed - expected_s) < 2e-6, f"{split_name}/{sensor}: dataset τ={observed} but expected {expected_s}"
    print(f"{split_name}: τ targets [ms] verified:", {sensor: round(float(state.time_offset.reshape(-1)[0]) * 1000, 3) for sensor, state in sample.current_calibration.items()})

# Compare the zero-clock and corrected-clock dataset instances on the same held-out
# physical windows. This checks whether the loader alters raw measurements when the
# rig reference changes; it never deliberately shifts raw timestamps in this notebook.
zero_clock_rig = KAISTRigCalibration(T_B_I=np.linalg.inv(T_I_B), T_B_L=T_B_L, tau_imu_s=0.0, tau_lidar_s=0.0, T_B_L_right=T_B_L_right, tau_lidar_right_s=0.0)
zero_clock_control = KAISTCalibrationDataset([spec_by_name[TEST_NAMES[0]]], zero_clock_rig, windowing_config, measurement_dtype=torch.float32, source_timestamp_dtype=torch.float64, window_timestamp_dtype=torch.float32)
corrected_control = KAISTCalibrationDataset([spec_by_name[TEST_NAMES[0]]], rig_calibration, windowing_config, measurement_dtype=torch.float32, source_timestamp_dtype=torch.float64, window_timestamp_dtype=torch.float32)
if len(zero_clock_control) != len(corrected_control):
    raise RuntimeError("Changing calibration τ modified the number of windows. Inspect dataset timestamp processing before training.")
for index in sorted({0, len(zero_clock_control)//2, len(zero_clock_control)-1}):
    old, new = zero_clock_control[index], corrected_control[index]
    for stream_name in old.streams:
        old_stream, new_stream = old.streams[stream_name], new.streams[stream_name]
        torch.testing.assert_close(old_stream.values, new_stream.values, rtol=0, atol=0)
        torch.testing.assert_close(old_stream.timestamps, new_stream.timestamps, rtol=0, atol=0)
        if old_stream.interval_start_timestamps is not None:
            torch.testing.assert_close(old_stream.interval_start_timestamps, new_stream.interval_start_timestamps, rtol=0, atol=0)
print("PASS: refined τ changes calibration reference while preserving native stream measurements and timestamps.")
del zero_clock_control, corrected_control


In [ ]:
train_shuffle_generator = torch.Generator().manual_seed(RANDOM_SEED)
train_loader = make_kaist_dataloader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=DROP_LAST_TRAIN, num_workers=NUM_WORKERS, generator=train_shuffle_generator)
val_loader = make_kaist_dataloader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=NUM_WORKERS)
test_loader = make_kaist_dataloader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=NUM_WORKERS)

print("train batches:", len(train_loader))
print("validation batches:", len(val_loader))
print("test batches:", len(test_loader))


## 4. Physically consistent truth augmentation, fine/coarse *independent* prior faults

In [ ]:
true_calibration_config = AugmentationConfig(
    sensor_topology=SensorTopologyAugmentationConfig(enabled=False),
    sampling_rate=SamplingRateAugmentationConfig(enabled=False),
    frame_randomization=FrameRandomizationConfig(enabled=False),
    common_body_frame=CommonBodyFrameRandomizationConfig(enabled=False, probability=0.0),
    true_calibration=TrueCalibrationAugmentationConfig(enabled=True, probability=1.0, rotation_probability=1.0, translation_probability=0.0, time_offset_probability=1.0, rotation=PerturbationMagnitudeConfig(maximum=TRUE_CALIBRATION_ROTATION_MAX_RAD), time_offset=PerturbationMagnitudeConfig(maximum=TRUE_CALIBRATION_TIME_OFFSET_MAX_S)),
    prior_perturbation=PriorPerturbationConfig(enabled=False),
    calibration_event=CalibrationEventConfig(enabled=False),
    noise=NoiseAugmentationConfig(enabled=False),
)
physical_augmenter = AugmentationPipeline(true_calibration_config)

def prior_config(rotation_max_rad, rotation_distribution, rotation_scale_rad=None):
    rotation_kwargs = dict(maximum=rotation_max_rad, distribution=rotation_distribution)
    if rotation_distribution == PerturbationDistribution.TRUNCATED_STUDENT_T:
        rotation_kwargs.update(scale=rotation_scale_rad, degrees_of_freedom=STUDENT_T_DOF)
    return PriorPerturbationConfig(
        enabled=True, probability=1.0,
        rotation=PerturbationMagnitudeConfig(**rotation_kwargs),
        translation=PerturbationMagnitudeConfig(maximum=PRIOR_TRANSLATION_MAX_M, distribution=PerturbationDistribution.TRUNCATED_STUDENT_T, scale=PRIOR_TRANSLATION_SCALE_M, degrees_of_freedom=STUDENT_T_DOF),
        time_offset=PerturbationMagnitudeConfig(maximum=PRIOR_TAU_MAX_S, distribution=PerturbationDistribution.TRUNCATED_STUDENT_T, scale=PRIOR_TAU_SCALE_S, degrees_of_freedom=STUDENT_T_DOF),
        one_calibration_per_sample=True, one_rotation_axis_per_sample=False, one_perturbation_type_per_sample=False,
    )

fine_prior_config = prior_config(np.deg2rad(FINE_ROTATION_MAX_DEG), PerturbationDistribution.TRUNCATED_STUDENT_T, np.deg2rad(FINE_ROTATION_SCALE_DEG))
coarse_prior_config = prior_config(np.deg2rad(COARSE_ROTATION_MAX_DEG), PerturbationDistribution.UNIFORM)
fine_prior_perturber = CalibrationPriorPerturber(fine_prior_config)
coarse_prior_perturber = CalibrationPriorPerturber(coarse_prior_config)


def augmentation_generator(seed):
    return torch.Generator(device=DEVICE.type).manual_seed(int(seed))


def clone_with_current_calibration(window, states):
    return WindowBatch(streams=dict(window.streams), current_calibration=states, metadata=dict(window.metadata), targets=window.targets, dataset_indices=window.dataset_indices)


def apply_independent_prior_faults(states, xi_by_key, tau_by_key):
    """Left-rotate only R; add independent parent-frame t and tau faults. Never rotate t with R."""
    updated = {}
    for key, state in states.items():
        xi = xi_by_key[key]
        prior = state.transform.clone()
        prior[:, :3, :3] = so3_exp(xi[:, :3]) @ state.transform[:, :3, :3]
        prior[:, :3, 3] = state.transform[:, :3, 3] + xi[:, 3:]
        updated[key] = replace(state, transform=prior, time_offset=state.time_offset + tau_by_key[key])
    return updated


@torch.no_grad()
def make_training_window(base_window, *, generator):
    """Physical augmentation followed by exactly one sensor's fine or coarse prior fault."""
    physical = physical_augmenter(base_window, generator=generator).augmented_window
    true_states = physical.current_calibration
    _, fine_xi, fine_tau = fine_prior_perturber(true_states, generator=generator)
    _, coarse_xi, coarse_tau = coarse_prior_perturber(true_states, generator=generator)
    batch_size = next(iter(true_states.values())).transform.shape[0]
    coarse_mask = torch.rand(batch_size, device=DEVICE, generator=generator) < COARSE_PRIOR_PROBABILITY
    # Uniform coarse angles across [60°,179°], not concentrated at small angles.
    coarse_angles = torch.deg2rad(torch.empty(batch_size, device=DEVICE, dtype=DTYPE).uniform_(COARSE_ROTATION_MIN_DEG, COARSE_ROTATION_MAX_DEG, generator=generator))
    xi, tau = {}, {}
    for key in true_states:
        current = coarse_xi[key].clone()
        angular_norm = torch.linalg.vector_norm(current[:, :3], dim=-1, keepdim=True)
        scaled_phi = current[:, :3] / angular_norm.clamp_min(1e-9) * coarse_angles[:, None]
        current[:, :3] = torch.where(angular_norm > 1e-8, scaled_phi, current[:, :3])
        xi[key] = torch.where(coarse_mask[:, None], current, fine_xi[key])
        tau[key] = torch.where(coarse_mask[:, None], coarse_tau[key], fine_tau[key])
    return clone_with_current_calibration(physical, apply_independent_prior_faults(true_states, xi, tau))


assert tuple(CALIBRATION_KEYS) == ('imu', 'lidar', 'lidar_right')
assert not true_calibration_config.sensor_topology.enabled
assert not true_calibration_config.frame_randomization.enabled
assert not true_calibration_config.common_body_frame.enabled
assert not true_calibration_config.noise.enabled
print('True-calibration re-expression enabled; topology/noise/frame augmentation disabled.')
print('Fine prior:', fine_prior_config)
print('Coarse prior:', coarse_prior_config)


## 5. Existing Transformer feature extractor + new 10-output regression head

The attached 6D patch must already exist under `src/obscalib/`. We use the same Transformer and zero-dimensional numeric calibration context as Notebook 47. Both the new head and all pretrained feature layers remain trainable. Only feature weights, not the old head, are loaded.

In [ ]:
model_config = PooledCalibrationModelConfig(
    transformer=TransformerConfig(input_dim=TOKEN_INPUT_DIM, d_model=D_MODEL, n_heads=N_HEADS, num_layers=NUM_TRANSFORMER_LAYERS, dim_feedforward=DIM_FEEDFORWARD, dropout=TRANSFORMER_DROPOUT, num_summary_tokens=NUM_SUMMARY_TOKENS, activation='gelu'),
    mlp=MLPConfig(hidden_dims=MLP_HIDDEN_DIMS, output_dim=SHARED_FEATURE_DIM, activation='gelu', dropout=MLP_DROPOUT),
    pooler=CalibrationPoolerConfig(n_heads=POOLER_N_HEADS, dropout=POOLER_DROPOUT, calibration_context_dim=0),
    head=PooledCalibrationHeadConfig(calibration_context_dim=0, hidden_dims=HEAD_HIDDEN_DIMS, activation='gelu', dropout=HEAD_DROPOUT),
)
geometry_processor = GeometryProcessor()
tokenizer = Tokenizer(measurement_dim=MEASUREMENT_DIM, calibration_keys=CALIBRATION_KEYS)
model = PooledCalibration6DModel(model_config).to(device=DEVICE, dtype=DTYPE)
if PRETRAIN_FEATURES:
    loaded_keys = load_6d_pretrained_features(model, SOURCE_FEATURE_CHECKPOINT, strict=True, trusted_checkpoint=True)
    print('Loaded pretrained feature tensors:', len(loaded_keys))
model.freeze_feature_extractor(False)
model.head_6d.requires_grad_(True)
assert all(p.requires_grad for p in model.parameters())


def model_predictions(window, *, model_override=None):
    """Reuse exactly the canonical GeometryProcessor/Tokenizer routing, no legacy SE3 updater."""
    if tuple(window.current_calibration) != CALIBRATION_KEYS:
        raise ValueError(f'Unexpected topology/order: {tuple(window.current_calibration)}')
    canonical_streams = geometry_processor(window.streams, window.metadata, window.current_calibration)
    tokens = tokenizer(canonical_streams, calibration_keys=tuple(window.current_calibration))
    return (model if model_override is None else model_override)(tokens)


def parameter_groups(module):
    transformer = list(module.transformer_encoder.parameters())
    other = list(module.pooler.parameters()) + list(module.mlp_body.parameters()) + list(module.head_6d.parameters())
    if len({id(p) for p in transformer + other}) != len(transformer + other):
        raise RuntimeError('Overlapping optimizer groups')
    if len(transformer + other) != len(list(module.parameters())):
        raise RuntimeError('Optimizer groups miss model parameters')
    return [dict(params=transformer, lr=TRANSFORMER_LR, name='transformer'), dict(params=other, lr=HEAD_MLP_LR, name='pooler_mlp_head')]


optimizer = torch.optim.AdamW(parameter_groups(model), weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=LR_REDUCE_FACTOR, patience=LR_PATIENCE_EPOCHS)
print('Parameters:', sum(p.numel() for p in model.parameters()), '| all trainable:', all(p.requires_grad for p in model.parameters()))
print('Learning rates:', [(group['name'], group['lr']) for group in optimizer.param_groups])


## 6. Full-range objective, differentiable updates and numerical diagnostics

In [ ]:
def rotation_geodesic_deg(R_first, R_second):
    """Numerically stable geodesic SO(3) angle, including near 180°. Shape [...]."""
    relative = R_first @ R_second.transpose(-1, -2)
    skew = torch.stack((relative[..., 2, 1] - relative[..., 1, 2], relative[..., 0, 2] - relative[..., 2, 0], relative[..., 1, 0] - relative[..., 0, 1]), dim=-1)
    sine = 0.5 * torch.linalg.vector_norm(skew, dim=-1)
    cosine = 0.5 * (relative.diagonal(dim1=-2, dim2=-1).sum(dim=-1) - 1.0)
    return torch.rad2deg(torch.atan2(sine, cosine))


def compute_6d_objective(window, output):
    """Joint 6D+chordal rotation loss, translation/tau loss, and active-fault emphasis."""
    if window.targets is None:
        raise ValueError('Training requires full GT targets')
    supervised = calibration_6d_loss(output.predictions, window.current_calibration, window.targets, weights=LOSS_WEIGHTS)
    active_terms, active_angle_errors, active_predicted_angles, active_target_angles = [], [], [], []
    for key in CALIBRATION_KEYS:
        state, truth, pred = window.current_calibration[key], window.targets[key], output.predictions[key]
        R_target = truth.next_transform[:, :3, :3] @ state.transform[:, :3, :3].transpose(-1, -2)
        angle_before = rotation_geodesic_deg(state.transform[:, :3, :3], truth.next_transform[:, :3, :3])
        active = angle_before > 1e-3
        if bool(active.any()):
            target_6d = matrix_to_rotation_6d(R_target)
            reg = (pred.rotation_6d - target_6d).square().mean(dim=-1)
            chord = 0.5 * (pred.delta_rotation - R_target).square().sum(dim=(-2, -1))
            active_terms.append((LOSS_WEIGHTS.rotation_6d_weight * reg + LOSS_WEIGHTS.rotation_chordal_weight * chord)[active])
            active_angle_errors.append(rotation_geodesic_deg(pred.delta_rotation[active], R_target[active]).detach())
            active_predicted_angles.append(rotation_geodesic_deg(pred.delta_rotation[active], torch.eye(3, device=DEVICE, dtype=DTYPE).expand_as(pred.delta_rotation[active])).detach())
            active_target_angles.append(angle_before[active].detach())
    extra_active = LOSS_WEIGHTS.rotation_weight * torch.cat(active_terms).mean() if active_terms else supervised.total * 0.0
    loss = supervised.total + ACTIVE_ROTATION_SUPERVISION_WEIGHT * extra_active
    with torch.no_grad():
        corrected = apply_calibration_6d_update(window.current_calibration, output.predictions)
        values = {}
        for key in CALIBRATION_KEYS:
            target = window.targets[key]
            state = corrected[key]
            values[f'{key}/rotation_error_deg'] = rotation_geodesic_deg(state.transform[:, :3, :3], target.next_transform[:, :3, :3]).mean().item()
            values[f'{key}/translation_error_cm'] = (100.0 * torch.linalg.vector_norm(state.transform[:, :3, 3] - target.next_transform[:, :3, 3], dim=-1)).mean().item()
            values[f'{key}/tau_error_ms'] = (1000.0 * (state.time_offset - target.next_time_offset).abs()).mean().item()
        values.update({'loss_total': float(loss.detach()), 'loss_supervised': float(supervised.total.detach()), 'loss_active_rotation': float(extra_active.detach()), 'loss_rotation_6d': float(supervised.rotation_6d.detach()), 'loss_rotation_chordal': float(supervised.rotation_chordal.detach()), 'loss_translation': float(supervised.translation.detach()), 'loss_tau': float(supervised.time_offset.detach()), 'active_rotation_error_deg': torch.cat(active_angle_errors).mean().item() if active_angle_errors else 0.0, 'active_target_deg': torch.cat(active_target_angles).mean().item() if active_target_angles else 0.0, 'active_predicted_step_deg': torch.cat(active_predicted_angles).mean().item() if active_predicted_angles else 0.0, 'active_rotation_count': sum(x.numel() for x in active_angle_errors)})
    return loss, values


def metric_average(sums, counts):
    return {key: sums[key] / counts[key] for key in sums}


## 7. Semantic and gradient smoke checks (including 179° target continuity)

In [ ]:
smoke_base = window_batch_to(collate_windows([train_dataset[0], train_dataset[min(100, len(train_dataset) - 1)]]), DEVICE, dtype=DTYPE)
smoke_window = make_training_window(smoke_base, generator=augmentation_generator(TRAIN_AUG_SEED + 101))
assert tuple(smoke_window.current_calibration) == CALIBRATION_KEYS
assert set(smoke_window.streams) == set(smoke_base.streams)
for key, stream in smoke_window.streams.items():
    original = smoke_base.streams[key]
    assert stream.values.shape == original.values.shape
    assert bool(torch.isfinite(stream.values).all())
    assert bool(torch.isfinite(stream.timestamps).all())
    torch.testing.assert_close(stream.sample_mask, original.sample_mask)
for key in CALIBRATION_KEYS:
    target = smoke_window.targets[key]
    assert target.next_transform is not None and target.next_time_offset is not None
    R = smoke_window.current_calibration[key].transform[:, :3, :3]
    R_gt = target.next_transform[:, :3, :3]
    R_delta = R_gt @ R.transpose(-1, -2)
    torch.testing.assert_close(rotation_6d_to_matrix(matrix_to_rotation_6d(R_delta)), R_delta, atol=5e-5, rtol=5e-5)

model.train()
optimizer.zero_grad(set_to_none=True)
smoke_output = model_predictions(smoke_window)
smoke_loss, smoke_metrics = compute_6d_objective(smoke_window, smoke_output)
smoke_loss.backward()
for part in ('transformer_encoder', 'pooler', 'mlp_body', 'head_6d'):
    grads = [p.grad for p in getattr(model, part).parameters() if p.requires_grad and p.grad is not None]
    assert grads and all(bool(torch.isfinite(g).all()) for g in grads), f'{part}: missing/nonfinite gradients'
    print(part, 'grad norm', round(math.sqrt(sum(float(g.square().sum()) for g in grads)), 5))
assert torch.isfinite(smoke_loss) and smoke_metrics['active_rotation_count'] > 0
optimizer.zero_grad(set_to_none=True)
model.eval()
print('PASS: 6D gradients reach all unfrozen layers; supervised full-GT update is valid.')
print('Smoke loss:', round(smoke_metrics['loss_total'], 4), '| active geodesic error:', round(smoke_metrics['active_rotation_error_deg'], 2), 'deg')
del smoke_base, smoke_window, smoke_output, smoke_loss


## 8. Fixed held-out angular probes and checkpoint score

Each probe repeats **the exact same augmented physical windows, rotation axes and prior faults** at every evaluation. Per-angle results are full-GT geodesic errors after *one uncapped 6D correction*, not velocity/CTG errors. The probe samples fault the two LiDARs while leaving other calibrations at GT to measure healthy-sensor drift. All methods see the same calibration convention.

In [ ]:
@torch.no_grad()
def build_probe_batches(dataset, *, n_windows, batch_size, seed, angles=PROBE_ANGLES_DEG, axis_seeds=PROBE_AXIS_SEEDS):
    indices = np.unique(np.linspace(0, len(dataset) - 1, min(len(dataset), n_windows)).round().astype(int))
    batches = []
    for axis_seed in range(axis_seeds):
        for group_id, start in enumerate(range(0, len(indices), batch_size)):
            chosen = indices[start:start + batch_size]
            base = window_batch_to(collate_windows([dataset[int(i)] for i in chosen]), DEVICE, dtype=DTYPE)
            generator = augmentation_generator(seed + axis_seed * 100003 + group_id * 1009)
            physical = physical_augmenter(base, generator=generator).augmented_window
            n = len(chosen)
            template = next(iter(physical.current_calibration.values())).transform
            axes = torch.randn(n, 3, generator=generator, device=template.device, dtype=template.dtype)
            axes = axes / torch.linalg.vector_norm(axes, dim=-1, keepdim=True).clamp_min(1e-9)
            cases = {}
            for angle in angles:
                xi = {key: torch.zeros(n, 6, device=DEVICE, dtype=DTYPE) for key in CALIBRATION_KEYS}
                tau = {key: torch.zeros(n, 1, device=DEVICE, dtype=DTYPE) for key in CALIBRATION_KEYS}
                keys = [PROBE_FAULT_KEYS[(j + axis_seed) % len(PROBE_FAULT_KEYS)] for j in range(n)]
                for j, key in enumerate(keys):
                    xi[key][j, :3] = axes[j] * np.deg2rad(float(angle))
                prior = apply_independent_prior_faults(physical.current_calibration, xi, tau)
                cases[float(angle)] = (clone_with_current_calibration(physical, prior), keys, chosen.copy(), axis_seed)
            batches.append(cases)
    return batches


@torch.inference_mode()
def evaluate_probe(model_to_evaluate, batches):
    original_training = model_to_evaluate.training
    model_to_evaluate.eval()
    rows = []
    try:
        for cases in batches:
            for angle, (window, fault_keys, indices, axis_seed) in cases.items():
                output = model_predictions(window, model_override=model_to_evaluate)
                updated = apply_calibration_6d_update(window.current_calibration, output.predictions)
                for j, active_key in enumerate(fault_keys):
                    pred_state, truth = updated[active_key], window.targets[active_key]
                    before = rotation_geodesic_deg(window.current_calibration[active_key].transform[j:j+1, :3, :3], truth.next_transform[j:j+1, :3, :3]).item()
                    after = rotation_geodesic_deg(pred_state.transform[j:j+1, :3, :3], truth.next_transform[j:j+1, :3, :3]).item()
                    step = rotation_geodesic_deg(output.predictions[active_key].delta_rotation[j:j+1], torch.eye(3, device=DEVICE, dtype=DTYPE).reshape(1, 3, 3)).item()
                    target_R = truth.next_transform[j:j+1, :3, :3] @ window.current_calibration[active_key].transform[j:j+1, :3, :3].transpose(-1, -2)
                    # Only a diagnostic. Geodesic error determines success, not signed axis agreement.
                    target_6d = matrix_to_rotation_6d(target_R)
                    embedded_error = (output.predictions[active_key].rotation_6d[j:j+1] - target_6d).square().mean().item()
                    healthy = [k for k in CALIBRATION_KEYS if k != active_key]
                    healthy_drift = np.mean([rotation_geodesic_deg(updated[k].transform[j:j+1, :3, :3], window.targets[k].next_transform[j:j+1, :3, :3]).item() for k in healthy])
                    rows.append(dict(angle_deg=float(angle), physical_index=int(indices[j]), axis_seed=axis_seed, fault_key=active_key, before_deg=before, after_deg=after, improvement_deg=before-after, relative_remaining=after/max(before, 1e-6) if before > 1e-4 else np.nan, step_deg=step, error_6d=embedded_error, healthy_drift_deg=healthy_drift, improved=after < before - 1e-5))
    finally:
        model_to_evaluate.train(original_training)
    return pd.DataFrame(rows)


def summarize_probe(frame):
    return frame.groupby('angle_deg', as_index=False).agg(n=('after_deg', 'size'), before_deg=('before_deg', 'mean'), final_deg=('after_deg', 'mean'), improvement_deg=('improvement_deg', 'mean'), relative_remaining=('relative_remaining', 'mean'), fraction_improving=('improved', 'mean'), predicted_step_deg=('step_deg', 'mean'), healthy_drift_deg=('healthy_drift_deg', 'mean'), error_6d=('error_6d', 'mean'))


def probe_score(summary):
    selected = summary[summary.angle_deg.isin(PROBE_SCORE_ANGLES_DEG)]
    if len(selected) != len(PROBE_SCORE_ANGLES_DEG):
        raise ValueError('Fixed probe has missing required angles')
    score = float(selected.relative_remaining.mean())
    zero = summary.loc[np.isclose(summary.angle_deg, 0.0), 'final_deg']
    zero_drift = float(zero.mean()) if len(zero) else float('nan')
    return score + ZERO_DRIFT_WEIGHT * zero_drift / 30.0


def print_probe_summary(frame, *, label):
    summary = summarize_probe(frame)
    print(label, '| normalized residual score:', f'{probe_score(summary):.4f}', '| lower is better')
    display(summary.round(3))
    return summary


full_probe_batches = build_probe_batches(val_dataset, n_windows=PROBE_WINDOWS, batch_size=PROBE_BATCH_SIZE, seed=VAL_AUG_SEED + 100000, axis_seeds=PROBE_AXIS_SEEDS)
preflight_probe_batches = build_probe_batches(val_dataset, n_windows=STOCHASTIC_PROBE_WINDOWS, batch_size=PROBE_BATCH_SIZE, seed=VAL_AUG_SEED + 200000, axis_seeds=1)
initial_probe_df = evaluate_probe(model, full_probe_batches)
initial_probe_summary = print_probe_summary(initial_probe_df, label='Initial pretrained features + fresh 6D head')


## 9. Reversible stochastic preflight, including 30°, 90°, 150° and 179° curves

In [ ]:
STOCHASTIC_SEED = TRAIN_AUG_SEED + 900000
stochastic_history = []
stochastic_probe_rows = []
STOCHASTIC_BEST_SCORE = float('inf')
STOCHASTIC_CANDIDATE_PATH = RUN_DIR / 'stochastic_best_6d_state.pt'

if RUN_STOCHASTIC_PRECHECK:
    saved_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    saved_training = model.training
    saved_cpu_rng = torch.get_rng_state().clone()
    saved_cuda_rng = torch.cuda.get_rng_state_all() if DEVICE.type == 'cuda' else None
    saved_np_rng, saved_py_rng = np.random.get_state(), random.getstate()
    pre_optimizer = torch.optim.AdamW(parameter_groups(model), weight_decay=WEIGHT_DECAY)
    pre_generator = augmentation_generator(STOCHASTIC_SEED)
    index_rng = np.random.default_rng(STOCHASTIC_SEED + 1)
    try:
        for step in range(STOCHASTIC_STEPS + 1):
            if step > 0:
                model.train()
                pre_optimizer.zero_grad(set_to_none=True)
                indices = index_rng.choice(len(train_dataset), size=min(STOCHASTIC_BATCH_SIZE, len(train_dataset)), replace=False)
                base = window_batch_to(collate_windows([train_dataset[int(i)] for i in indices]), DEVICE, dtype=DTYPE)
                batch = make_training_window(base, generator=pre_generator)
                output = model_predictions(batch)
                loss, metrics = compute_6d_objective(batch, output)
                loss.backward()
                grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
                pre_optimizer.step()
                train_loss = float(loss.detach())
                del base, batch, output, loss
            else:
                train_loss, grad_norm = float('nan'), float('nan')
            if step == 0 or step % STOCHASTIC_REPORT_EVERY == 0 or step == STOCHASTIC_STEPS:
                frame = evaluate_probe(model, preflight_probe_batches)
                frame['step'] = step
                stochastic_probe_rows.extend(frame.to_dict('records'))
                summary = summarize_probe(frame)
                score = probe_score(summary)
                stochastic_history.append(dict(step=step, train_loss=train_loss, grad_norm=float(grad_norm), probe_score=score, **{f'angle_{int(a)}_deg':float(summary.loc[summary.angle_deg == a,'final_deg'].iloc[0]) for a in PROBE_ANGLES_DEG}))
                if score < STOCHASTIC_BEST_SCORE:
                    STOCHASTIC_BEST_SCORE = score
                    if SAVE_STOCHASTIC_CANDIDATE:
                        torch.save({'model_state_dict':{key:value.detach().cpu().clone() for key,value in model.state_dict().items()}, 'step':step, 'score':score}, STOCHASTIC_CANDIDATE_PATH)
                selection = {int(a):float(summary.loc[summary.angle_deg==a,'final_deg'].iloc[0]) for a in (30.0, 90.0, 150.0, 179.0)}
                print(f'step {step:4d}/{STOCHASTIC_STEPS} | train={train_loss:.3f} grad={float(grad_norm):.2f} | score={score:.4f} | 30/90/150/179° residuals={selection}', flush=True)
    finally:
        model.load_state_dict(saved_state, strict=True)
        model.train(saved_training)
        optimizer.zero_grad(set_to_none=True)
        torch.set_rng_state(saved_cpu_rng)
        if saved_cuda_rng is not None:
            torch.cuda.set_rng_state_all(saved_cuda_rng)
        np.random.set_state(saved_np_rng)
        random.setstate(saved_py_rng)
        del saved_state, pre_optimizer
        gc.collect()
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
    print('Preflight restored the original weights and RNG. Main optimizer remains untouched.')
else:
    print('Stochastic preflight disabled.')


In [ ]:
if stochastic_history:
    pd.DataFrame(stochastic_history).to_csv(RUN_DIR / 'stochastic_6d_history.csv', index=False)
    pd.DataFrame(stochastic_probe_rows).to_csv(RUN_DIR / 'stochastic_6d_angle_cases.csv', index=False)
    trace = pd.DataFrame(stochastic_history)
    fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
    axes[0].plot(trace.step, trace.train_loss, label='Stochastic train loss')
    ax0 = axes[0].twinx()
    ax0.plot(trace.step, trace.probe_score, color='C1', label='Fixed probe score')
    axes[0].set(xlabel='Optimizer step', ylabel='Train loss', title='Stochastic objective and held-out score')
    ax0.set_ylabel('Normalized residual score')
    h1,l1=axes[0].get_legend_handles_labels();h2,l2=ax0.get_legend_handles_labels();axes[0].legend(h1+h2,l1+l2,fontsize=8)
    for a in PROBE_ANGLES_DEG:
        if a == 0:
            continue
        axes[1].plot(trace.step, trace[f'angle_{int(a)}_deg'], marker='.', label=f'{int(a)}°')
    axes[1].set(xlabel='Optimizer step', ylabel='Remaining GT rotation error [deg]', title='Same validation cases across stochastic steps')
    axes[1].legend(ncol=2,fontsize=8)
    for ax in axes:ax.grid(alpha=.25)
    fig.tight_layout();fig.savefig(RUN_DIR / 'stochastic_6d_learning.png', dpi=170)
    if SHOW_PLOTS:plt.show()
    else:plt.close(fig)


## 10. Full-training epoch runner and checkpoint initialization

All weights are unfrozen from epoch 1. `ReduceLROnPlateau` responds to **validation supervised loss**. The **best checkpoint** is selected by a fixed-angle, angle-balanced normalized residual score (30°, 90°, 150°, 175°, 179° plus 0° drift), while both metrics are logged. Fixed angular validation uses untouched held-out KAIST sequences, not the test split.

In [ ]:
def run_6d_epoch(loader, *, training, generator, max_batches=None):
    model.train(training)
    sums, counts = defaultdict(float), defaultdict(float)
    n_samples, optimizer_updates = 0, 0
    pending_samples, pending_microbatches = 0, 0
    started = time.perf_counter()
    if training:
        optimizer.zero_grad(set_to_none=True)

    def optimizer_flush():
        nonlocal pending_samples, pending_microbatches, optimizer_updates
        if pending_samples == 0:
            return
        for p in model.parameters():
            if p.grad is not None:
                p.grad.div_(pending_samples)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRADIENT_CLIP_NORM)
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
        optimizer_updates += 1
        pending_samples, pending_microbatches = 0, 0

    with (torch.enable_grad() if training else torch.no_grad()):
        for batch_id, cpu_batch in enumerate(loader):
            if max_batches is not None and batch_id >= max_batches:
                break
            base = window_batch_to(cpu_batch, DEVICE, dtype=DTYPE)
            augmented = make_training_window(base, generator=generator)
            prediction = model_predictions(augmented)
            loss, metrics = compute_6d_objective(augmented, prediction)
            n = len(next(iter(augmented.current_calibration.values())).transform)
            if training:
                (loss * n).backward()
                pending_samples += n
                pending_microbatches += 1
                if pending_microbatches >= GRADIENT_ACCUMULATION_STEPS:
                    optimizer_flush()
            for key, value in metrics.items():
                if key == 'active_rotation_count':
                    continue
                sums[key] += float(value) * n
                counts[key] += n
            n_samples += n
            del base, augmented, prediction, loss
        if training:
            optimizer_flush()
    if not n_samples:
        raise RuntimeError('Epoch used no samples')
    result = metric_average(sums, counts)
    result.update(n_samples=n_samples, optimizer_updates=optimizer_updates, elapsed_s=time.perf_counter()-started)
    return result


BEST_CHECKPOINT_PATH, LAST_CHECKPOINT_PATH = RUN_DIR / 'best.pt', RUN_DIR / 'last.pt'
HISTORY_PATH = RUN_DIR / 'training_6d_history.csv'
checkpoint_config = dict(run_name=RUN_NAME, architecture='PooledCalibration6DModel', calibration_keys=CALIBRATION_KEYS, source_checkpoint=str(SOURCE_FEATURE_CHECKPOINT) if PRETRAIN_FEATURES else None, head_mlp_lr=HEAD_MLP_LR, transformer_lr=TRANSFORMER_LR, batch_size=BATCH_SIZE, gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS, loss_weights=asdict(LOSS_WEIGHTS), active_weight=ACTIVE_ROTATION_SUPERVISION_WEIGHT, fine_max_deg=FINE_ROTATION_MAX_DEG, coarse_range_deg=(COARSE_ROTATION_MIN_DEG, COARSE_ROTATION_MAX_DEG), coarse_probability=COARSE_PRIOR_PROBABILITY, translation_prior_m=PRIOR_TRANSLATION_MAX_M, tau_reference_s=REFINED_TAU_BY_SENSOR_S, train_names=tuple(TRAIN_NAMES), val_names=tuple(VAL_NAMES), probe_angles=PROBE_ANGLES_DEG, probe_windows=PROBE_WINDOWS, probe_score_angles=PROBE_SCORE_ANGLES_DEG, zero_drift_weight=ZERO_DRIFT_WEIGHT)
train_generator = augmentation_generator(TRAIN_AUG_SEED)
start_epoch, global_step, best_score, best_epoch = 0, 0, float('inf'), 0
history=[]
if RESUME_FULL_TRAINING:
    if not LAST_CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(LAST_CHECKPOINT_PATH)
    metadata = torch.load(LAST_CHECKPOINT_PATH, map_location='cpu', weights_only=False)
    extra = metadata.get('extra_state') or {}
    if extra.get('config_6d') != checkpoint_config:
        raise RuntimeError('6D resume configuration differs; use a new RUN_NAME rather than resuming')
    loaded = load_training_checkpoint(LAST_CHECKPOINT_PATH, model=model, optimizer=optimizer, scheduler=scheduler, generators={'train_shuffle': train_shuffle_generator, 'train_aug': train_generator}, map_location=DEVICE, restore_rng_state=True)
    start_epoch, global_step = int(loaded.get('epoch', 0)), int(loaded.get('global_step', 0))
    best_score, best_epoch = float(extra['best_probe_score']), int(extra.get('best_epoch',0))
    if HISTORY_PATH.is_file():
        history=pd.read_csv(HISTORY_PATH).to_dict('records')
    print('Resumed exact 6D run at epoch', start_epoch)
else:
    if RUN_FULL_TRAINING and any(p.exists() for p in (BEST_CHECKPOINT_PATH, LAST_CHECKPOINT_PATH, HISTORY_PATH)):
        raise FileExistsError('Run files already exist; choose a new RUN_NAME or RESUME_FULL_TRAINING=True')
    if LOAD_STOCHASTIC_CANDIDATE_FOR_FULL:
        if not STOCHASTIC_CANDIDATE_PATH.is_file():
            raise FileNotFoundError(STOCHASTIC_CANDIDATE_PATH)
        candidate = torch.load(STOCHASTIC_CANDIDATE_PATH, map_location='cpu', weights_only=True)
        model.load_state_dict(candidate['model_state_dict'], strict=True)
        print('Explicitly initialized from stochastic candidate step', candidate['step'])
    else:
        print('Initialized full run from pretrained features and fresh 6D head (preflight restored).')


def save_6d_checkpoint(path, epoch, metrics):
    save_training_checkpoint(path, model=model, optimizer=optimizer, scheduler=scheduler, epoch=epoch, global_step=global_step, metrics=metrics, generators={'train_shuffle':train_shuffle_generator, 'train_aug':train_generator}, extra_state={'config_6d': checkpoint_config, 'best_probe_score':best_score, 'best_epoch':best_epoch})


## 11. Full training with 30° / 90° / 150° fixed-angle reporting *every epoch*

In [ ]:
if RUN_FULL_TRAINING:
    for epoch in range(start_epoch, NUM_EPOCHS):
        train_shuffle_generator.manual_seed(RANDOM_SEED + 12001 + epoch)
        train = run_6d_epoch(train_loader, training=True, generator=train_generator, max_batches=MAX_TRAIN_BATCHES_PER_EPOCH)
        val = run_6d_epoch(val_loader, training=False, generator=augmentation_generator(VAL_AUG_SEED), max_batches=MAX_VAL_BATCHES_PER_EPOCH)
        scheduler.step(val['loss_total'])
        global_step += train['optimizer_updates']
        row = dict(epoch=epoch+1, global_step=global_step, train_loss=train['loss_total'], val_loss=val['loss_total'], train_rotation_6d=train['loss_rotation_6d'], val_rotation_6d=val['loss_rotation_6d'], train_rotation_chordal=train['loss_rotation_chordal'], val_rotation_chordal=val['loss_rotation_chordal'], train_translation=train['loss_translation'], val_translation=val['loss_translation'], train_tau=train['loss_tau'], val_tau=val['loss_tau'], train_active_deg=train['active_rotation_error_deg'], val_active_deg=val['active_rotation_error_deg'], head_mlp_lr=optimizer.param_groups[1]['lr'], transformer_lr=optimizer.param_groups[0]['lr'])
        if (epoch+1) % PROBE_EVERY_EPOCHS == 0:
            probe = evaluate_probe(model, full_probe_batches)
            summary = summarize_probe(probe)
            score = probe_score(summary)
            row['probe_score'] = score
            for angle in PROBE_ANGLES_DEG:
                sub=summary.loc[summary.angle_deg==angle].iloc[0]
                row[f'probe_{int(angle)}_final_deg'] = float(sub.final_deg)
                row[f'probe_{int(angle)}_improves'] = float(sub.fraction_improving)
            if score < best_score:
                best_score, best_epoch = score, epoch+1
                save_6d_checkpoint(BEST_CHECKPOINT_PATH, epoch+1, row)
                print('New best angular probe checkpoint at epoch', best_epoch, '| score',round(score,4))
        history.append(row)
        pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)
        save_6d_checkpoint(LAST_CHECKPOINT_PATH, epoch+1, row)
        angles_report=', '.join(f'{int(a)}°={row[f"probe_{int(a)}_final_deg"]:.1f}°' for a in (30.0,90.0,150.0,175.0,179.0)) if 'probe_score' in row else 'not evaluated'
        print(f'Epoch {epoch+1:3d}/{NUM_EPOCHS} train={train["loss_total"]:.4f} val={val["loss_total"]:.4f} | fixed probe: {angles_report} | best score={best_score:.4f} (epoch {best_epoch}) | LR={row["head_mlp_lr"]:.1e}/{row["transformer_lr"]:.1e}', flush=True)
else:
    print('Full training disabled.')


## 12. Full-run learning curves and per-angle history

In [ ]:
if HISTORY_PATH.is_file():
    train_history = pd.read_csv(HISTORY_PATH)
    fig, axes = plt.subplots(1, 3, figsize=(19, 4.5))
    axes[0].plot(train_history.epoch, train_history.train_loss, label='Train')
    axes[0].plot(train_history.epoch, train_history.val_loss, label='Validation')
    axes[0].set(xlabel='Epoch',ylabel='Supervised objective',title='6D regression training')
    axes[1].plot(train_history.epoch, train_history.train_rotation_6d, label='6D train')
    axes[1].plot(train_history.epoch, train_history.val_rotation_6d, label='6D val')
    axes[1].plot(train_history.epoch, train_history.val_rotation_chordal, label='Chordal val')
    axes[1].set(xlabel='Epoch',ylabel='Rotation loss',title='Rotation objective components')
    axes[2].plot(train_history.epoch, train_history.probe_score, label='Angular validation score')
    axes[2].set(xlabel='Epoch',ylabel='Normalized residual + drift penalty',title='Fixed held-out probe')
    for ax in axes:ax.grid(alpha=.25);ax.legend(fontsize=8)
    fig.tight_layout();fig.savefig(RUN_DIR / '6d_training_objectives.png',dpi=170)
    if SHOW_PLOTS:plt.show()
    else:plt.close(fig)

    fig, axes=plt.subplots(1, 2, figsize=(14, 4.5))
    for a in PROBE_ANGLES_DEG:
        axes[0].plot(train_history.epoch, train_history[f'probe_{int(a)}_final_deg'],label=f'{int(a)}°')
    axes[0].set(xlabel='Epoch',ylabel='Remaining error after one correction [deg]',title='Held-out per-angle errors')
    for a in PROBE_ANGLES_DEG:
        if a == 0:continue
        axes[1].plot(train_history.epoch, train_history[f'probe_{int(a)}_improves'],label=f'{int(a)}°')
    axes[1].set(xlabel='Epoch',ylabel='Fraction improved',ylim=(-0.03,1.03),title='Held-out correction success rate')
    for ax in axes:ax.grid(alpha=.25);ax.legend(ncol=2,fontsize=8)
    fig.tight_layout();fig.savefig(RUN_DIR / '6d_training_angles.png',dpi=170)
    if SHOW_PLOTS:plt.show()
    else:plt.close(fig)
else:
    print('No full training history yet.')


## 13. Best-checkpoint versus initial model, paired on identical faults

The baseline is the **same pretrained encoder with the identity-initialized 6D head** (not the legacy 7D model). This isolates what the 6D training learned. A separate comparison against older regressors/FM can be added afterward using their respective prediction/update adapters.

In [ ]:
if BEST_CHECKPOINT_PATH.is_file():
    model.eval()
    previous = {key:value.detach().cpu().clone() for key,value in model.state_dict().items()}
    try:
        load_training_checkpoint(BEST_CHECKPOINT_PATH, model=model, map_location=DEVICE, restore_rng_state=False)
        best_probe_df = evaluate_probe(model, full_probe_batches)
        best_probe_summary = print_probe_summary(best_probe_df, label='Best 6D checkpoint')
        paired = initial_probe_df.merge(best_probe_df, on=['angle_deg','physical_index','axis_seed','fault_key'], suffixes=('_initial','_trained'),validate='one_to_one')
        paired['error_gain_deg'] = paired.after_deg_initial - paired.after_deg_trained
        print('Paired residual reduction (positive is better):')
        display(paired.groupby('angle_deg').error_gain_deg.agg(['mean','median','std','count']).round(3))
        best_probe_df.to_csv(RUN_DIR/'best_6d_angle_cases.csv',index=False)
        paired.to_csv(RUN_DIR/'paired_initial_vs_trained_6d.csv',index=False)
        fig,axes=plt.subplots(1,2,figsize=(14,4.5))
        for label,sub in [('Initial',initial_probe_summary),('Trained',best_probe_summary)]:
            axes[0].plot(sub.angle_deg,sub.final_deg,marker='o',label=label)
            axes[1].plot(sub.angle_deg,sub.fraction_improving,marker='o',label=label)
        axes[0].plot(PROBE_ANGLES_DEG,PROBE_ANGLES_DEG,linestyle=':',color='gray',label='No correction')
        axes[0].set(xlabel='Initial rotation error [deg]',ylabel='After one update [deg]',title='Full-range regression, paired angles')
        axes[1].set(xlabel='Initial rotation error [deg]',ylabel='Fraction improving',ylim=(-0.03,1.03),title='Paired success rate')
        for ax in axes:ax.grid(alpha=.25);ax.legend()
        fig.tight_layout();fig.savefig(RUN_DIR/'best_6d_angle_comparison.png',dpi=170)
        if SHOW_PLOTS:plt.show()
        else:plt.close(fig)
    finally:
        model.load_state_dict(previous,strict=True)
        del previous
else:
    print('No best checkpoint available yet.')


## 14. Optional untouched test-split evaluation (explicitly off by default)

In [ ]:
if RUN_FINAL_TEST_EVALUATION:
    if not BEST_CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(BEST_CHECKPOINT_PATH)
    previous = {key:value.detach().cpu().clone() for key,value in model.state_dict().items()}
    try:
        load_training_checkpoint(BEST_CHECKPOINT_PATH, model=model, map_location=DEVICE, restore_rng_state=False)
        test_probe_batches = build_probe_batches(test_dataset, n_windows=PROBE_WINDOWS, batch_size=PROBE_BATCH_SIZE, seed=TEST_AUG_SEED + 100000, axis_seeds=PROBE_AXIS_SEEDS)
        test_probe_frame = evaluate_probe(model, test_probe_batches)
        print_probe_summary(test_probe_frame, label='FINAL TEST: 6D checkpoint')
        test_probe_frame.to_csv(RUN_DIR / 'final_test_6d_angle_cases.csv',index=False)
    finally:
        model.load_state_dict(previous,strict=True)
        del previous
else:
    print('Test split untouched. Set RUN_FINAL_TEST_EVALUATION=True only for the final comparison.')


## 15. Interpretation and next decisions

- **Do not judge by training loss alone.** Prefer the paired fixed validation probe at 30°, 90°, 150°, 175° and 179°, and the zero-error drift at 0°.
- If the stochastic objective falls but 150–179° error does not, changing LR alone is unlikely to address global rotational identifiability. Compare an optional fixed-batch overfit test before changing architecture again.
- The new 6D head does **not** guarantee coarse-angle success. It removes the principal-log discontinuity from the correction output, while the `GeometryProcessor` continues to map relative SO(3)/SE(3) *measurements* through logarithms after expressing them under the current calibration. Those inputs are relative-motion logs, not full correction targets.
- The stochastic probe restores the initialization; the full run starts from it unless `LOAD_STOCHASTIC_CANDIDATE_FOR_FULL=True`.
- This notebook uses unbounded *rotation correction* supervision, but applies one update per 5-second physical window. Its inference cost is one Transformer forward plus Gram–Schmidt, not an FM ODE integration.
- **Next research control:** compare against a Wahba/SVD angular-alignment feature branch on the same validation windows, especially if the 6D regressor still fails at 175–179°.

In [ ]:
print('Run:', RUN_DIR)
print('New model: PooledCalibration6DModel | 6 rotation + 3 translation + 1 tau')
print('Pretrained feature checkpoint:', SOURCE_FEATURE_CHECKPOINT if PRETRAIN_FEATURES else 'none')
print('All trainable:', all(p.requires_grad for p in model.parameters()))
print('Optimizer LRs:', [(group['name'], group['lr']) for group in optimizer.param_groups])
print('Best checkpoint:', BEST_CHECKPOINT_PATH, '| exists:', BEST_CHECKPOINT_PATH.is_file())
print('Training history:', HISTORY_PATH)
